# 03 · Análisis de Progenitores del ICL y BCG
**Metodología:** §7 de Mayes+2026

1. **§7.1** – Masa media de progenitores ICL vs BCG (Fig. 13)
2. **§7.2** – Fracción de progenitores compartidos ICL↔BCG (Fig. 14)
3. **§7.3** – Localización de material de progenitores alta/baja masa (Fig. 15)
4. **§7.4** – Progenitor más significativo por componente


In [ ]:
import sys, os, pickle
import numpy as np
import h5py
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM
from scipy.stats import linregress
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, './original_shift_code')
import illustris_python as il
import Catalogue
import params_icl as P

# Carpetas de salida para las figuras de este notebook
FIG_PDF = './figuras/03_analisis_progenitores/pdf'
FIG_PNG = './figuras/03_analisis_progenitores/png'
os.makedirs(FIG_PDF, exist_ok=True)
os.makedirs(FIG_PNG, exist_ok=True)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 12,
                     'axes.spines.top': False, 'axes.spines.right': False})

cosmo = FlatLambdaCDM(H0=67.74, Om0=0.3089)
Header = il.groupcat.loadHeader(P.basePath, P.SNAP)

with h5py.File(P.CATALOG_OUT, 'r') as f:
    group_idx   = f['group_idx'][:]
    M200c       = f['M200c_Msun'][:]
    GroupPos    = f['GroupPos_kpc'][:]
    bcg_sub_idx = f['bcg_sub_idx'][:]
    dyn_state   = f['dyn_state'][:] if 'dyn_state' in f else None

n_cl = len(group_idx)
lM   = np.log10(M200c)
COLORS_STATE = {0:'#2196F3', 1:'#FF9800', 2:'#F44336'}
LABELS_STATE = {0:'Relajado', 1:'Intermedio', 2:'Perturbado'}
print(f"Catálogo cargado: {n_cl} cúmulos")

In [2]:
# Catálogo SA – estadísticas AGREGADAS por subhalo (ver read_hdf5_SA.ipynb)
with h5py.File(P.SA_FILE, 'r') as f:
    s99 = f['Snapshot_99']
    sa_insitu    = s99['StellarMassInSitu'][:]                    * P.UM
    sa_exsitu    = s99['StellarMassExSitu'][:]                    * P.UM
    sa_mergers   = s99['StellarMassFromCompletedMergers'][:]      * P.UM
    sa_maj_comp  = s99['StellarMassFromCompletedMergersMajor'][:] * P.UM
    sa_min_comp  = s99['StellarMassFromCompletedMergersMajorMinor'][:] * P.UM - sa_maj_comp
    sa_ongoing   = s99['StellarMassFromOngoingMergers'][:]        * P.UM
    sa_maj_ong   = s99['StellarMassFromOngoingMergersMajor'][:]   * P.UM
    sa_total     = s99['StellarMassTotal'][:]                     * P.UM

print(f"Catálogo SA: {len(sa_total):,} subhalos")

# Fracciones para los BCGs
sa_tot_bcg = sa_total[bcg_sub_idx]
ok_m = sa_tot_bcg > 0
f_insitu  = np.where(ok_m, sa_insitu[bcg_sub_idx]  / sa_tot_bcg, np.nan)
f_exsitu  = np.where(ok_m, sa_exsitu[bcg_sub_idx]  / sa_tot_bcg, np.nan)
f_mergers = np.where(ok_m, sa_mergers[bcg_sub_idx] / sa_tot_bcg, np.nan)
f_ongoing = np.where(ok_m, sa_ongoing[bcg_sub_idx] / sa_tot_bcg, np.nan)
f_maj_c   = np.where(ok_m, sa_maj_comp[bcg_sub_idx]/ sa_tot_bcg, np.nan)
f_min_c   = np.where(ok_m, sa_min_comp[bcg_sub_idx]/ sa_tot_bcg, np.nan)
f_maj_o   = np.where(ok_m, sa_maj_ong[bcg_sub_idx] / sa_tot_bcg, np.nan)

print(f"Mediana f_insitu  = {np.nanmedian(f_insitu):.3f}")
print(f"Mediana f_mergers = {np.nanmedian(f_mergers):.3f}  (completados)")
print(f"Mediana f_ongoing = {np.nanmedian(f_ongoing):.3f}  (stripped)")


Catálogo SA: 4,371,211 subhalos
Mediana f_insitu  = 0.297
Mediana f_mergers = 0.597  (completados)
Mediana f_ongoing = 0.000  (stripped)


In [3]:
# Funciones auxiliares (mismas que notebook 02)
def rotate_by_inertia_tensor(pos_rel, mass, r_lim=np.inf):
    dist = np.linalg.norm(pos_rel, axis=1)
    ok   = (dist>0)&(dist<=r_lim)&np.isfinite(mass)
    p,m  = pos_rel[ok], mass[ok]
    if m.sum()==0 or len(m)<4: return pos_rel, np.eye(3)
    w = 1/dist[ok]**2; mt = m.sum()
    Ixx=np.sum(m*p[:,0]**2*w)/mt; Iyy=np.sum(m*p[:,1]**2*w)/mt
    Izz=np.sum(m*p[:,2]**2*w)/mt; Ixy=np.sum(m*p[:,0]*p[:,1]*w)/mt
    Ixz=np.sum(m*p[:,0]*p[:,2]*w)/mt; Iyz=np.sum(m*p[:,1]*p[:,2]*w)/mt
    I=np.array([[Ixx,Ixy,Ixz],[Ixy,Iyy,Iyz],[Ixz,Iyz,Izz]])
    ev,evec=np.linalg.eigh(I)
    R=evec[:,np.argsort(ev)[::-1]].T
    return pos_rel@R.T, R

def sb_and_holmberg(r_2d, lum_r, n_bins=60):
    r_bins=np.logspace(np.log10(0.5),np.log10(np.percentile(r_2d,99)),n_bins+1)
    r_mid=np.sqrt(r_bins[:-1]*r_bins[1:]); mu_r=np.full(n_bins,np.nan)
    for k,(r1,r2) in enumerate(zip(r_bins[:-1],r_bins[1:])):
        mk=(r_2d>=r1)&(r_2d<r2)
        if mk.sum()==0: continue
        sl=lum_r[mk].sum()/(np.pi*((r2*1e3)**2-(r1*1e3)**2))
        if sl>0: mu_r[k]=P.SB_CONST-2.5*np.log10(sl)
    valid=np.isfinite(mu_r)&(r_mid>0)
    r_h=np.nan
    if valid.sum()>=3:
        rv,mv=r_mid[valid],mu_r[valid]
        idx=np.argsort(rv); rv,mv=rv[idx],mv[idx]
        if mv[0]<=P.MU_HOLMBERG<=mv[-1]:
            try: r_h=float(interp1d(mv,rv,fill_value='extrapolate')(P.MU_HOLMBERG))
            except: pass
    return r_h

def hmr(r, m):
    if len(r)==0 or m.sum()==0: return np.nan
    idx=np.argsort(r); cum=np.cumsum(m[idx])
    i50=np.searchsorted(cum,cum[-1]/2)
    return r[idx[min(i50,len(r)-1)]]

def linfit(x, y):
    ok=np.isfinite(x)&np.isfinite(y)
    if ok.sum()<3: return np.nan,np.nan
    sl,ic,*_=linregress(x[ok],y[ok]); return sl,ic

def load_region_data(sub_id, cen_pos, header=Header):
    """Carga partículas estelares, aplica corte de Holmberg. Sin SA por-partícula."""
    fields=['Coordinates','Masses','GFM_StellarPhotometrics']
    st=il.snapshot.loadSubhalo(P.basePath,P.SNAP,int(sub_id),'stars',fields=fields)
    pos  =Catalogue.Distance_3D(st['Coordinates']*P.UL, cen_pos, header['BoxSize']*P.UL)
    mass =st['Masses']*P.UM
    phot =st['GFM_StellarPhotometrics']
    pos_rot,_=rotate_by_inertia_tensor(pos,mass)
    r_2d=np.sqrt(pos_rot[:,0]**2+pos_rot[:,1]**2)
    lum_r=10**((P.M_SUN_R_AB-phot[:,5])/2.5)
    r_h=sb_and_holmberg(r_2d,lum_r)
    mask_bcg = r_2d<=r_h if np.isfinite(r_h) else np.zeros(len(r_2d),bool)
    return {'mass':mass,'r_2d':r_2d,'lum_r':lum_r,
            'mask_bcg':mask_bcg,'mask_icl':~mask_bcg,'r_h':r_h}

def get_merger_history(sub_id):
    """
    Carga el árbol SubLink y extrae masas estelares de progenitores.
    Devuelve array con masa estelar de cada satélite que se fusionó con el BCG.
    La masa reportada es la del snapshot más reciente del satélite en el árbol.
    """
    fields = ['SubhaloMassType', 'SnapNum', 'FirstProgenitorID', 'NextProgenitorID',
              'DescendantID']
    try:
        tree = il.sublink.loadTree(P.basePath, P.SNAP, int(sub_id),
                                    fields=fields, onlyMPB=False)
    except Exception:
        return np.array([])
    if tree is None or len(tree['SnapNum']) == 0:
        return np.array([])

    # Identificar MPB: seguir FirstProgenitorID desde el índice 0
    n = len(tree['SnapNum'])
    in_mpb = np.zeros(n, dtype=bool)
    in_mpb[0] = True
    ptr = tree['FirstProgenitorID'][0]
    while ptr >= 0 and ptr < n:
        in_mpb[ptr] = True
        ptr = tree['FirstProgenitorID'][ptr]

    # Los progenitores fuera del MPB son satélites que se fusionaron
    sat_stars = tree['SubhaloMassType'][~in_mpb, 4] * P.UM
    return sat_stars[sat_stars > 1e7]  # >1e7 M☉ para excluir ruido numérico


## §7.1 – Distribución de masas de progenitores desde el árbol SubLink (Fig. 13)

El catálogo SA da fracciones TOTALES del subhalo central (BCG+ICL juntos).
Para separar qué progenitores contribuyeron al BCG vs al ICL se necesitaría un SA
por-partícula, que no está disponible aquí.

**Alternativa implementada**: usamos el árbol de fusiones SubLink para identificar
los satélites que se fusionaron con el BCG y obtener su masa estelar al momento de la
fusión. Esto da la **distribución de masas de progenitores para el BCG+ICL total**.
El desglose mayor/menor se toma del catálogo SA (campos `CompletedMergersMajor`).


In [4]:
print("§7.1 – Masa de progenitores desde SubLink...")
mean_pm  = np.full(n_cl, np.nan)   # masa media de progenitores [M☉]
frac_hi  = np.full(n_cl, np.nan)   # fracción de masa ex situ de prog. > M_PROG_THRESH
n_sat    = np.full(n_cl, np.nan)   # número de satélites que contribuyeron

for i in range(n_cl):
    try:
        sat_m = get_merger_history(bcg_sub_idx[i])
    except Exception as e:
        print(f"  Error cúmulo {i}: {e}"); continue
    if len(sat_m) == 0: continue
    n_sat[i]   = len(sat_m)
    mean_pm[i] = np.average(sat_m)
    m_tot      = sat_m.sum()
    hi_mask    = sat_m >= P.M_PROG_THRESH
    frac_hi[i] = sat_m[hi_mask].sum() / m_tot if m_tot > 0 else np.nan
    if (i+1)%10==0: print(f"  {i+1}/{n_cl}", end="\r")
print("\nListo.")
print(f"Masa media progenitores: {np.nanmedian(mean_pm):.2e} M☉")
print(f"Fracción de prog. M>1e10 M☉: {np.nanmean(frac_hi):.2f} ± {np.nanstd(frac_hi):.2f}  (paper: 0.65±0.15)")
print(f"N medio de satélites fusionados: {np.nanmedian(n_sat):.0f}")


§7.1 – Masa de progenitores desde SubLink...
  160/166
Listo.
Masa media progenitores: 1.08e+10 M☉
Fracción de prog. M>1e10 M☉: 0.96 ± 0.02  (paper: 0.65±0.15)
N medio de satélites fusionados: 2012


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,5))

# Panel izq: masa media progenitores vs M200c
ax = axes[0]
ok = np.isfinite(mean_pm)
sc = ax.scatter(lM[ok], np.log10(mean_pm[ok]), c=lM[ok], cmap='viridis', s=25, alpha=0.8)
plt.colorbar(sc, ax=ax, label=r'$\log M_{{200c}}$')
sl, ic = linfit(lM[ok], np.log10(mean_pm[ok]))
if np.isfinite(sl):
    xx = np.linspace(lM.min(), lM.max(), 100)
    ax.plot(xx, sl*xx+ic, 'k--', lw=1.5, label=f'β={sl:.3f}')
ax.set_xlabel(r'$\log M_{{200c}}$')
ax.set_ylabel(r'$\log\langle M_{\rm prog}\rangle\;[M_\odot]$')
ax.set_title('Masa media progenitores (análogo Fig. 13)'); ax.legend()

# Panel der: fracción de alta masa vs M200c (comparar con SA mayor/menor)
ax = axes[1]
for f_arr, col, lbl in [(frac_hi, 'tomato', 'SubLink: M>10¹⁰ M☉'),
                         (f_maj_c, '#9C27B0', 'SA: fusión mayor')]:
    ok = np.isfinite(f_arr)
    ax.scatter(lM[ok], f_arr[ok], color=col, label=lbl, s=20, alpha=0.7)
    sl, ic = linfit(lM[ok], f_arr[ok])
    if np.isfinite(sl):
        xx = np.linspace(lM.min(), lM.max(), 100)
        ax.plot(xx, sl*xx+ic, color=col, lw=1.5, ls='--')
ax.set_xlabel(r'$\log M_{{200c}}$')
ax.set_ylabel('Fracción de masa')
ax.set_title('Progenitores de alta masa (SubLink vs SA)'); ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIG_PDF}/fig13_masa_progenitores.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_PNG}/fig13_masa_progenitores.png', bbox_inches='tight', dpi=150)
plt.show()

## §7.2 – Fracciones de ensamblaje del catálogo SA vs propiedades del halo (Fig. 8/9 equivalent)

Los §7.2 (progenitores compartidos ICL↔BCG) y §7.3 (HMR por masa de progenitor en BCG vs ICL)
de Mayes+2026 requieren un catálogo SA por-partícula para separar componentes espacialmente.
Con el catálogo agregado disponible, mostramos las correlaciones entre las fracciones de
ensamblaje y las propiedades del halo, que equivalen a los resultados de nivel más alto.


In [ ]:
fig, axes = plt.subplots(2,2,figsize=(13,10))

# SA fracciones vs M200c
for ax, f_arr, col, lbl in [
    (axes[0,0], f_insitu,  '#E91E63', 'In situ'),
    (axes[0,1], f_exsitu,  '#2196F3', 'Ex situ total'),
    (axes[1,0], f_maj_c,   '#9C27B0', 'Fusiones mayores completadas'),
    (axes[1,1], f_maj_o,   '#00BCD4', 'Stripped de fusiones mayores'),
]:
    ok = np.isfinite(f_arr)
    ax.scatter(lM[ok], f_arr[ok], color=col, s=20, alpha=0.7)
    sl, ic = linfit(lM[ok], f_arr[ok])
    if np.isfinite(sl):
        xx = np.linspace(lM.min(), lM.max(), 100)
        ax.plot(xx, sl*xx+ic, color=col, lw=1.8, ls='--', label=f'β={sl:.3f}')
    if dyn_state is not None:
        for s, col_s, lbl_s in [(0,'#2196F3','Rel.'),(1,'#FF9800','Int.'),(2,'#F44336','Pert.')]:
            ms = (dyn_state==s)&ok
            ax.scatter(lM[ms], f_arr[ms], color=col_s, label=lbl_s, s=15, alpha=0.9, zorder=5)
    ax.set_xlabel(r'$\log M_{{200c}}$'); ax.set_ylabel('Fracción de masa')
    ax.set_title(lbl); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{FIG_PDF}/fig_sa_fracciones.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_PNG}/fig_sa_fracciones.png', bbox_inches='tight', dpi=150)
plt.show()

## §7.3 – Historia de ensamblaje a lo largo del tiempo (multi-snapshot SA)

In [ ]:
# Fracción in situ a diferentes redshifts para trazar la historia de ensamblaje
# El catálogo SA tiene datos en todos los snapshots
snap_list = [20, 33, 50, 67, 80, 91, 99]  # z≈3, 2, 1, 0.5, 0.2, 0.05, 0
z_list    = []

insitu_t  = np.full((n_cl, len(snap_list)), np.nan)
exsitu_t  = np.full((n_cl, len(snap_list)), np.nan)

with h5py.File(P.SA_FILE, 'r') as f:
    for j, sn in enumerate(snap_list):
        key = f'Snapshot_{sn}'
        if key not in f: continue
        m_in  = f[key]['StellarMassInSitu'][:]  * P.UM
        m_tot = f[key]['StellarMassTotal'][:] * P.UM
        for i in range(n_cl):
            si = bcg_sub_idx[i]
            if si < len(m_tot) and m_tot[si] > 0:
                insitu_t[i,j]  = m_in[si]  / m_tot[si]
                exsitu_t[i,j]  = 1 - insitu_t[i,j]

# Calcular z para cada snapshot
from astropy.cosmology import FlatLambdaCDM
cosmo = FlatLambdaCDM(H0=67.74, Om0=0.3089)
header_snaps = {}
for sn in snap_list:
    try:
        h_s = il.groupcat.loadHeader(P.basePath, sn)
        z_list.append(1/h_s['Time'] - 1)
    except:
        z_list.append(np.nan)
z_arr = np.array(z_list)

fig, ax = plt.subplots(figsize=(9,6))
ax.plot(z_arr, np.nanmedian(insitu_t, axis=0), 'r-o', lw=2, label='In situ (mediana)')
ax.fill_between(z_arr,
    np.nanpercentile(insitu_t,16,axis=0),
    np.nanpercentile(insitu_t,84,axis=0),
    color='red', alpha=0.2)
ax.plot(z_arr, np.nanmedian(exsitu_t, axis=0), 'b-s', lw=2, label='Ex situ (mediana)')
ax.fill_between(z_arr,
    np.nanpercentile(exsitu_t,16,axis=0),
    np.nanpercentile(exsitu_t,84,axis=0),
    color='blue', alpha=0.2)
ax.set_xlabel('Redshift z'); ax.set_ylabel('Fracción de masa estelar')
ax.set_title('Evolución del ensamblaje estelar (BCG+ICL total)')
ax.legend(); ax.invert_xaxis()
plt.tight_layout()
plt.savefig(f'{FIG_PDF}/fig_evolucion_ensamblaje.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_PNG}/fig_evolucion_ensamblaje.png', bbox_inches='tight', dpi=150)
plt.show()

## §7.4 – Progenitor más significativo (árbol SubLink)


In [8]:
print("§7.4 – Progenitor más significativo...")
frac_top1 = np.full(n_cl, np.nan)
n_for_90  = np.full(n_cl, np.nan)

for i in range(n_cl):
    try:
        sat_m = get_merger_history(bcg_sub_idx[i])
    except Exception:
        continue
    if len(sat_m) == 0: continue
    sorted_m = np.sort(sat_m)[::-1]
    m_tot = sorted_m.sum()
    if m_tot == 0: continue
    frac_top1[i] = sorted_m[0] / m_tot
    cumf = np.cumsum(sorted_m) / m_tot
    n_for_90[i] = np.searchsorted(cumf, 0.9) + 1
    if (i+1)%10==0: print(f"  {i+1}/{n_cl}", end="\r")
print("\nListo.")
print(f"Contribución top progenitor: {np.nanmean(frac_top1):.3f}±{np.nanstd(frac_top1):.3f}  (paper ICL: 0.27±0.12)")
print(f"N progenitores para 90%: {np.nanmedian(n_for_90):.0f}")


§7.4 – Progenitor más significativo...
  160/166
Listo.
Contribución top progenitor: 0.017±0.003  (paper ICL: 0.27±0.12)
N progenitores para 90%: 126


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,5))

ax = axes[0]
bins = np.linspace(0,1,25)
ax.hist(frac_top1[np.isfinite(frac_top1)], bins=bins, color='royalblue', alpha=0.8)
ax.axvline(np.nanmedian(frac_top1), color='royalblue', lw=2, ls='--',
           label=f'Mediana={np.nanmedian(frac_top1):.3f}')
ax.set_xlabel('Fracción del progenitor más significativo')
ax.set_ylabel('N cúmulos'); ax.set_title('BCG+ICL: progenitor más significativo'); ax.legend()

ax = axes[1]
ok = np.isfinite(n_for_90)
sc = ax.scatter(lM[ok], n_for_90[ok], c=lM[ok], cmap='viridis', s=25, alpha=0.8)
plt.colorbar(sc, ax=ax, label=r'$\log M_{{200c}}$')
sl, ic = linfit(lM[ok], n_for_90[ok])
if np.isfinite(sl):
    xx = np.linspace(lM.min(), lM.max(), 100)
    ax.plot(xx, sl*xx+ic, 'k--', lw=1.5, label=f'β={sl:.3f}')
ax.set_xlabel(r'$\log M_{{200c}}$')
ax.set_ylabel('N progenitores para el 90% del BCG+ICL')
ax.set_title('Progenitores necesarios para 90% de masa'); ax.legend()

plt.tight_layout()
plt.savefig(f'{FIG_PDF}/fig_progenitor_significativo.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_PNG}/fig_progenitor_significativo.png', bbox_inches='tight', dpi=150)
plt.show()

## Comparación con Mayes+2026

| Resultado | Mayes+2026 | Este análisis | Método |
|-----------|-----------|---------------|--------|
| f_insitu BCG+ICL | ~50% | *(ver fig 8)* | SA agregado |
| f_exsitu BCG+ICL | ~50% | *(ver fig 8)* | SA agregado |
| Fracción de prog. M>10¹⁰ M☉ | 65 ± 15% | *(ver §7.1)* | SubLink |
| Contribución top progenitor | **0.27 ± 0.12** | *(ver §7.4)* | SubLink |
| Progenitores compartidos ICL↔BCG | **93%** | *requiere SA por-partícula* | — |
| HMR por masa de progenitor | Fig. 15 | *requiere SA por-partícula* | — |

**Nota**: Los análisis de Figs. 14 y 15 de Mayes+2026 que separan BCG vs ICL a nivel de
progenitor individual requieren un catálogo SA por-partícula con `ProgGalaxyID`.
El catálogo SA disponible (`stellar_assembly.hdf5`) es agregado por subhalo.
